# Variantes de DRAEM
Cada defecto generado requiere la tapa normal original de TRAIN, la imagen alterada, la máscara alineada y una revisión humana de calidad. El objetivo reconstructivo sigue siendo la tapa original.

In [ ]:
from collections import defaultdict
import random


def synthetic_pairs(frame):
    base = DATA / "synthetic"
    pairs = pd.read_csv(base / "pairs.csv", dtype=str, keep_default_na=False)
    normal = frame[(frame.split == "train") & (frame.label == 0)].set_index("file")
    if pairs.empty or not pairs.review.eq("accepted").all():
        raise ValueError("El conjunto sintético necesita pares aprobados.")
    mapping = defaultdict(list)
    evidence_rows = []
    for pair in pairs.itertuples():
        if (
            pair.source not in normal.index
            or not pair.generator.strip()
            or not pair.defect.strip()
        ):
            raise ValueError("Revisa la procedencia, generador y tipo de defecto.")
        clean = normal.at[pair.source, "path"]
        altered, mask = [(base / value).resolve() for value in (pair.image, pair.mask)]
        if not all(p.is_relative_to(base.resolve()) for p in (altered, mask)):
            raise ValueError("Los pares deben estar dentro de data/synthetic.")
        with (
            Image.open(clean) as original,
            Image.open(altered) as generated,
            Image.open(mask) as target,
        ):
            binary = np.asarray(target.convert("L"))
            if (
                original.size != generated.size
                or original.size != target.size
                or not binary.any()
                or not set(np.unique(binary)) <= {0, 1, 255}
            ):
                raise ValueError(f"Máscara o tamaño inválido en {pair.image}.")
        mapping[str(clean)].append((altered, mask))
        evidence_rows.append(
            [pair.source, digest(altered), digest(mask), pair.generator, pair.defect]
        )
    if len({row[1] for row in evidence_rows}) != len(pairs):
        raise ValueError("Hay imágenes sintéticas repetidas.")
    return (
        mapping,
        fingerprint(evidence_rows),
        pairs.groupby(["defect", "generator"]).size(),
    )

## Aprendizaje con pares de imágenes

In [ ]:
class PairedDraem(Draem):
    def training_step(self, batch, *args, **kwargs):
        clean = batch.image
        pairs = self.pairs.get(batch.image_path[0], [])
        if self.variant == "draem-mixed" and random.random() < 0.5:
            augmented, mask = self.augmenter(clean)
        elif pairs and random.random() < 0.5:
            image_path, mask_path = random.choice(pairs)
            with Image.open(image_path) as image, Image.open(mask_path) as source:
                image = tv_tensors.Image(
                    pil_to_tensor(image.convert("RGB")).float() / 255
                )
                target = tv_tensors.Mask(pil_to_tensor(source.convert("L"))[0] > 0)
                image, target = self.pair_transform(image, target)
            augmented = image.unsqueeze(0).to(clean)
            mask = target.unsqueeze(0).unsqueeze(0).to(clean)
        else:
            augmented = clean
            mask = torch.zeros_like(clean[:, :1])
        restored, prediction = self.model(augmented)
        return self.loss(clean, restored, mask, prediction)